In [1]:
from __future__ import annotations

import argparse
import importlib.util
import logging
import sys
import time
import types
from pathlib import Path
from tqdm import tqdm

from src.utils import pmf_utils
import pandas as pd
import matplotlib.pyplot as plt
import importlib, src.ddm.ddm
importlib.reload(src.ddm.ddm)

import numpy as np
import torch
import pickle

from config import dir_config

from src.ddm.utils import build_stimulus, get_job, build_grid, prepare_data, load_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [2]:
# change directory to project root
import os
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

from scripts.ddm.ddm_fitting import DDMModel

In [3]:
processed_dir = Path(dir_config.data.processed)
ddm_dir = processed_dir / 'ddm'
# ddm_dir = processed_dir / 'ddm_5a9ce9d'

session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")
behavior_df = pd.read_csv(ddm_dir / "behavior_data.csv")

In [4]:
# find all folders in ddm_dir
model_folders = [f for f in ddm_dir.iterdir() if f.is_dir()]
print("DDM folders found:", model_folders)

DDM folders found: [PosixPath('/mnt/prior-data/processed/ddm/leak-1_tc-0'), PosixPath('/mnt/prior-data/processed/ddm/plain_ddm'), PosixPath('/mnt/prior-data/processed/ddm/leak-0_tc-0'), PosixPath('/mnt/prior-data/processed/ddm/leak-1_tc-1'), PosixPath('/mnt/prior-data/processed/ddm/leak-0_tc-1')]


In [5]:
session_ids = session_metadata["session_id"].tolist()
grid = build_grid(behavior_df)
job_lookup = {job_id: get_job(grid, job_id) for job_id in range(len(grid))}
stem_to_job_id = {
    (
        job["session_id"],
        job["prior_block"],
        job["enable_leak"],
        job["enable_time_constant"],
        job["enable_sv"],
        job["enable_sz"],
    ): job_id
    for job_id, job in job_lookup.items()
}

MODEL_VARIANTS = [
    {"name": "leak-0_tc-0", "enable_leak": False, "enable_time_constant": False, "enable_sv": True,  "enable_sz": True},
    {"name": "leak-0_tc-1", "enable_leak": False, "enable_time_constant": True,  "enable_sv": True,  "enable_sz": True},
    {"name": "leak-1_tc-0", "enable_leak": True,  "enable_time_constant": False, "enable_sv": True,  "enable_sz": True},
    {"name": "leak-1_tc-1", "enable_leak": True,  "enable_time_constant": True,  "enable_sv": True,  "enable_sz": True},
    {"name": "plain_ddm",   "enable_leak": False, "enable_time_constant": False, "enable_sv": False, "enable_sz": False},
]

missing_job_ids = set()

for variant in MODEL_VARIANTS:
    sub_dir = ddm_dir / variant["name"]
    leak = variant["enable_leak"]
    tc   = variant["enable_time_constant"]
    sv   = variant["enable_sv"]
    sz   = variant["enable_sz"]

    expected = {
        (sid, block, leak, tc, sv, sz)
        for sid in session_ids
        for block in [0, 1]
    }

    found = set()
    for model in sub_dir.rglob("*.pkl"):
        try:
            session = model.stem.split("_prior_block_")[0]
            block   = int(model.stem.split("_prior_block_")[1])
            found.add((session, block, leak, tc, sv, sz))
        except Exception as e:
            print(f"[PARSE ERROR] {model}: {e}")

    missing = expected - found
    extra   = found - expected

    print(
        f"\n=== {variant['name']} ===\n"
        f"Total files: {len(list(sub_dir.rglob('*.pkl')))}\t"
        f"Expected: {len(expected)}\t"
        f"Missing: {len(missing)}\t"
        f"Extra: {len(extra)}"
    )

    if missing:
        print("\nMISSING DETAILS:")
        for m in sorted(missing):
            job_id = stem_to_job_id.get(m)
            print(f"{m} → job_id={job_id}")
            if job_id is not None:
                missing_job_ids.add(job_id)

print("\n=== FITTING SUMMARY ===")
for variant in MODEL_VARIANTS:
    model_path = ddm_dir / variant["name"]
    failed = list(model_path.glob("*.FAILED.json"))
    if failed:
        print(f"Model {variant['name']} has {len(failed)} failed fits:")
        for f in failed:
            print(f"  {f.name}")
    else:
        print(f"Model {variant['name']} has no failed fits.")

print("\n=== SUMMARY ===")
print("Missing job IDs:", sorted(missing_job_ids))


=== leak-0_tc-0 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-0_tc-1 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-1_tc-0 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-1_tc-1 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== plain_ddm ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== FITTING SUMMARY ===
Model leak-0_tc-0 has no failed fits.
Model leak-0_tc-1 has no failed fits.
Model leak-1_tc-0 has no failed fits.
Model leak-1_tc-1 has no failed fits.
Model plain_ddm has no failed fits.

=== SUMMARY ===
Missing job IDs: []


# Session Wise Model Fits

In [7]:
# calculate cumulative likelihood for all model types
cumulative_likelihoods = {}
aic_values = {}
bic_values = {}

# free params: ndt, a, z, drift_gain, drift_offset (+sv, +sz, +leak_rate, +time_constant where enabled)
PARAM_COUNTS = {
    "leak-0_tc-0": 7,  # base 5 + sv + sz
    "leak-0_tc-1": 8,  # + time_constant
    "leak-1_tc-0": 8,  # + leak_rate
    "leak-1_tc-1": 9,  # + leak_rate + time_constant
    "plain_ddm":   5,  # base 5 only
}

for variant in MODEL_VARIANTS:
    model_type = variant["name"]
    model_path = ddm_dir / model_type

    pkl_files = list(model_path.glob("*.pkl"))

    total_nll = 0
    for pkl_file in tqdm(pkl_files, desc=f"Loading {model_type} models", unit="file"):
        try:
            model = load_model(pkl_file)
            total_nll += model['results']['likelihood']
        except Exception as e:
            print(f"Error loading {pkl_file}: {e}")

    k = PARAM_COUNTS[model_type]
    cumulative_likelihoods[model_type] = np.round(total_nll)
    aic_values[model_type] = 2 * k + 2 * np.round(total_nll)
    bic_values[model_type] = np.round(k * np.log(len(behavior_df)) + 2 * np.round(total_nll))

best_likelihood_model = min(cumulative_likelihoods, key=cumulative_likelihoods.get)
best_aic_model        = min(aic_values,             key=aic_values.get)
best_bic_model        = min(bic_values,             key=bic_values.get)

if best_likelihood_model == best_aic_model == best_bic_model:
    print(f"Best model by likelihood, AIC, and BIC: {best_likelihood_model}")
else:
    print(f"Best model by likelihood: {best_likelihood_model}")
    print(f"Best model by AIC:        {best_aic_model}")
    print(f"Best model by BIC:        {best_bic_model}")

Loading plain_ddm models: 100%|██████████| 90/90 [01:03<00:00,  1.41file/s]

Best model by likelihood, AIC, and BIC: leak-1_tc-1


In [8]:
print("=== MODEL COMPARISON ===")
print(f"{'Model':<15} {'Cumulative NLL':<20} {'AIC':<10} {'BIC':<10}")
for model in cumulative_likelihoods.keys():
    print(f"{model:<15} {cumulative_likelihoods[model]:<20} {aic_values[model]:<10} {bic_values[model]:<10}")


=== MODEL COMPARISON ===
Model           Cumulative NLL       AIC        BIC       
leak-0_tc-0     6335.0               12684.0    12745.0   
leak-0_tc-1     5983.0               11982.0    12052.0   
leak-1_tc-0     6284.0               12584.0    12654.0   
leak-1_tc-1     5968.0               11954.0    12033.0   
plain_ddm       7201.0               14412.0    14456.0   
